# Quantum Fast Weight Programmers (QFWP)

**What you'll learn:**
* Why fast-weight programming is an alternative to explicit recurrence for sequence models
* How a parameterized quantum circuit can act as the *programmer* that emits weight updates for a classical *slow* network
* How the fast-weight update rule is applied and trained end-to-end with PyTorch
* Where cuQuantum fits: simulating the programmer circuit and batching its evaluations across a sequence on the GPU
* What to profile before scaling the model up

> **Status:** this notebook is a scaffold. Section structure, imports, and interfaces are in place; cells marked `TODO` are completed by the presenter for the live session.

> **Where you are: Afternoon Block, Notebook 02 — Quantum Sequence Models.**
> The morning block trained a classical model *around* a quantum circuit. This notebook inverts that relationship: a small parameterized quantum circuit *reprograms* the weights of a classical network, giving a sequence model without an explicit recurrent hidden state. The circuit stays small enough for state-vector simulation, so cuQuantum is the natural execution path; a CPU path remains available for portability, with those timings read as functional only.

## Notebook Topic Map

| Topic | Section | Hands-on result |
|:---|:---|:---|
| Fast weights versus recurrence | **1. Background** | Understand what the programmer / slow-network split buys you |
| The quantum programmer | **2. Ansatz and update rule** | Build the circuit and the weight-modulation step |
| Training and scaling | **3-4. Sequence task** | Train end-to-end and profile where the time actually goes |

## 0. Environment and Imports

The scaffold imports cleanly with or without a GPU. `cuquantum` is brought in behind a guard so the notebook can be opened and read on a CPU-only machine; the programmer circuit itself requires it at run time.

In [ ]:
import numpy as np
import torch
import torch.nn as nn

# cuQuantum provides the simulation backend for the programmer circuit.
# Guarded so this scaffold still imports on a CPU-only machine.
try:
    import cuquantum
    HAS_CUQUANTUM = True
except ImportError:
    cuquantum = None
    HAS_CUQUANTUM = False

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device      : {device}")
print(f"cuQuantum available: {HAS_CUQUANTUM}")

## 1. Fast Weight Programmers in One Page

A recurrent network carries context in a hidden state. A *fast weight programmer* carries it in the weights instead: a **programmer** network observes each input and emits an update to a second **slow** network's weight matrix, so the slow network is literally reprogrammed as the sequence advances.

**Why this shape suits a quantum circuit:**

- **The programmer stays small.** It emits a low-rank correction rather than a full weight matrix, so a circuit with a handful of qubits can drive a much larger classical layer.
- **There is no recurrent state to simulate.** Context lives in the classical weights, so the quantum part remains a fixed-depth circuit evaluated once per step instead of an ever-deeper one.
- **Capacity scales with depth, not width.** Adding expressivity means more circuit layers, not a wider hidden state — which is exactly the direction that stays tractable under state-vector simulation.

## 2. The Quantum Fast Weight Programmer

### 2.1 Programmer circuit ansatz

The programmer is a data re-uploading circuit: the input embedding is encoded as rotation angles, interleaved with trainable rotation layers and a ring of entangling gates. Reading out per-qubit Pauli-Z expectations yields a real vector, which becomes the fast-weight update in the next section.

In [ ]:
# TODO(Samuel): implement the programmer ansatz.
#
# Contract: given trainable angles `theta` of shape (n_layers, n_qubits, 3)
# and an input embedding `x` of length n_qubits, prepare the data
# re-uploading state and return the per-qubit Pauli-Z expectation values as
# a real vector of length n_qubits. Those expectations are the raw material
# for the fast-weight update in section 2.2.
#
# Backend: build the statevector with cuQuantum so that a batch of parameter
# sets can be evaluated in a single call rather than one circuit at a time.

def qfwp_programmer(theta: np.ndarray, x: np.ndarray) -> np.ndarray:
    """Evaluate the programmer circuit and return <Z_i> for each qubit."""
    raise NotImplementedError("Programmer ansatz to be provided by the presenter.")

### 2.2 Applying the update to the slow network

The programmer output modulates the slow weights rather than overwriting them. Keeping a base matrix and applying a correction on top is what lets context accumulate across a sequence while each individual update stays low-rank and cheap to apply.

In [ ]:
# TODO(Samuel): implement the fast-weight cell.
#
# The slow network holds a base weight W0. At each step the programmer emits
# a low-rank correction that modulates W0 instead of replacing it, so context
# accumulates across the sequence without an explicit recurrent hidden state.

class QFWPCell(nn.Module):
    def __init__(self, d_in: int, d_out: int, n_qubits: int, n_layers: int):
        super().__init__()
        # W0    : (d_out, d_in)             base "slow" weights
        # theta : (n_layers, n_qubits, 3)   programmer circuit parameters
        raise NotImplementedError("Fast-weight cell to be provided by the presenter.")

    def forward(self, x_seq: torch.Tensor) -> torch.Tensor:
        """x_seq: (batch, seq_len, d_in) -> returns (batch, seq_len, d_out)"""
        raise NotImplementedError

## 3. Training on a Sequence Task

The training loop is ordinary PyTorch — the circuit sits inside the forward pass and is differentiated along with everything else:

1. Sample a batch of sequences from the task.
2. Roll the cell over the sequence, accumulating fast-weight updates step by step.
3. Compute the task loss on the final prediction.
4. Backpropagate into both the programmer angles and the slow weights.

In [ ]:
from dataclasses import dataclass


@dataclass
class CFG:
    n_qubits: int = 4
    n_layers: int = 2
    d_in: int = 8
    d_out: int = 8
    seq_len: int = 32
    batch_size: int = 16
    lr: float = 1e-3
    epochs: int = 20
    seed: int = 0


# TODO(Samuel): training loop over the chosen sequence task.
# Defaults above are sized for a live demo, not for a published result.

def train_qfwp(cfg: CFG):
    raise NotImplementedError("Training loop to be provided by the presenter.")

## 4. Scaling Notes

- **Where the time goes:** the programmer circuit is evaluated once per step, so at these qubit counts the classical optimizer step, not the circuit, usually dominates. Profile both before deciding what to accelerate.
- **Batching the programmer:** evaluating a batch of parameter sets in a single cuQuantum call amortizes launch overhead across the sequence, which is what makes longer sequences practical in a classroom setting.
- **Read fallback runs honestly:** on a CPU-only environment the scaffold still imports and the classical parts still run, but any timing collected there measures the fallback rather than GPU-accelerated simulation, and should not be compared against cuQuantum numbers.

## References

* Self-Modulating Quantum Fast-Weight Programmers for Efficient Adaptive Sequential Learning — [arXiv:2606.24933](https://arxiv.org/abs/2606.24933)
* Stable Self-Modulating Quantum Fast-Weight Programmers with Bounded Memory Gates — [arXiv:2607.02363](https://arxiv.org/abs/2607.02363)
* J. Schmidhuber, *Learning to Control Fast-Weight Memories* (1992) — the classical fast-weight formulation
* cuQuantum SDK: [docs.nvidia.com/cuda/cuquantum](https://docs.nvidia.com/cuda/cuquantum/latest/index.html)

---
## Coming up next — Notebook 03

QFWP used a quantum circuit to *generate* the parameters of a classical model. The next two notebooks keep the same scaling question but change the mechanism:

| Notebook | What it does |
|:---|:---|
| `03_qkan_basics.ipynb` | **Quantum-inspired** Kolmogorov–Arnold Networks: data re-uploading activations replace parameter-heavy MLP blocks, with a cuTensorNet solver path. |
| `04_cutn-qsvm.ipynb` | **Quantum-enhanced** SVM: a quantum feature-map kernel captured with `cudaq_einsum` and contracted in batch via cuTensorNet. |

See you in the next notebook. 🚀